> **Disclaimer:** This workshop is provided for educational and informational purposes only. The sample code and configurations are not intended for production use. Please review and adapt them according to your organization's security and compliance requirements. AWS service charges may apply for resources created during this workshop. Video content used in this workshop is sourced from publicly available materials and is attributed where applicable.

import json
import time
import boto3
import numpy as np
from botocore.config import Config

# Configuration - Update these values for your environment
REGION = "us-east-1"
MODEL_ID = "twelvelabs.marengo-embed-3-0-v1:0"
MODEL_ID_SYNC = "us.twelvelabs.marengo-embed-3-0-v1:0"
S3_BUCKET = "<YOUR-S3-BUCKET>"                  # e.g. "my-video-workshop-bucket"
AWS_ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
VIDEO_S3_KEY = "input/SPOTV2.mp4"               # Update with your video path
VIDEO_S3_URI = f"s3://{S3_BUCKET}/{VIDEO_S3_KEY}"
OUTPUT_S3_PREFIX = "output/marengo-multivector/"
DATA_S3_PREFIX = f"{OUTPUT_S3_PREFIX}embeddings/"

# Clients with SigV4 for S3 credentials
bedrock = boto3.client("bedrock-runtime", region_name=REGION, config=Config(signature_version='v4'))
s3 = boto3.client("s3", region_name=REGION)

print("Configuration ready")
print(f"  Model: {MODEL_ID}")
print(f"  Video: {VIDEO_S3_URI}")
print(f"  Output: s3://{S3_BUCKET}/{OUTPUT_S3_PREFIX}")
print(f"  Data: s3://{S3_BUCKET}/{DATA_S3_PREFIX}")

In [38]:
import json
import time
import boto3
import numpy as np
from botocore.config import Config

# Configuration - Update these values for your environment
REGION = "us-east-1"
MODEL_ID = "twelvelabs.marengo-embed-3-0-v1:0"
MODEL_ID_SYNC = "us.twelvelabs.marengo-embed-3-0-v1:0"
S3_BUCKET = "twelvelabs-video-demo-bucket"                  # e.g. "my-video-workshop-bucket"
AWS_ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
VIDEO_S3_KEY = "videos/Sparks_4096x2160_5994fps_SDR.mp4"
VIDEO_S3_URI = f"s3://{S3_BUCKET}/{VIDEO_S3_KEY}"
OUTPUT_S3_PREFIX = "output/marengo-multivector/"
DATA_S3_PREFIX = f"{OUTPUT_S3_PREFIX}embeddings/"
VECTOR_BUCKET = "video-vectors-087967639890" 

# Clients with SigV4 for S3 credentials
bedrock = boto3.client("bedrock-runtime", region_name=REGION, config=Config(signature_version='v4'))
s3 = boto3.client("s3", region_name=REGION)

print("Configuration ready")
print(f"  Model: {MODEL_ID}")
print(f"  Video: {VIDEO_S3_URI}")
print(f"  Output: s3://{S3_BUCKET}/{OUTPUT_S3_PREFIX}")
print(f"  Data: s3://{S3_BUCKET}/{DATA_S3_PREFIX}")

Configuration ready
  Model: twelvelabs.marengo-embed-3-0-v1:0
  Video: s3://twelvelabs-video-demo-bucket/videos/Sparks_4096x2160_5994fps_SDR.mp4
  Output: s3://twelvelabs-video-demo-bucket/output/marengo-multivector/
  Data: s3://twelvelabs-video-demo-bucket/output/marengo-multivector/embeddings/


<cell_type>markdown</cell_type>## 1. Verify Video Exists in S3

In [26]:
# Check if video exists in S3
try:
    response = s3.head_object(Bucket=S3_BUCKET, Key=VIDEO_S3_KEY)
    size_mb = response['ContentLength'] / (1024 * 1024)
    print(f"✅ Video exists in S3: {VIDEO_S3_URI}")
    print(f"   Size: {size_mb:.2f} MB")
    print(f"   Last Modified: {response['LastModified']}")
except Exception as e:
    print(f"❌ Video not found: {e}")
    print(f"   Please upload your video to {VIDEO_S3_URI}")
    raise

✅ Video exists in S3: s3://twelvelabs-video-demo-bucket/videos/Sparks_4096x2160_5994fps_SDR.mp4
   Size: 400.30 MB
   Last Modified: 2026-04-01 09:20:10+00:00


## 2. Generate Multi-Vector Embeddings
Request visual, audio, and transcription embeddings with clip and asset scope using dynamic segmentation.

In [27]:
response = bedrock.start_async_invoke(
    modelId=MODEL_ID,
    modelInput={
        "inputType": "video",
        "video": {
            "mediaSource": {
                "s3Location": {
                    "uri": VIDEO_S3_URI,
                    "bucketOwner": AWS_ACCOUNT_ID
                }
            },
            "embeddingOption": ["visual", "audio", "transcription"],
            "embeddingScope": ["clip", "asset"],
            "segmentation": {
                "method": "dynamic",
                "dynamic": {"minDurationSec": 4}
            }
        }
    },
    outputDataConfig={
        "s3OutputDataConfig": {
            "s3Uri": f"s3://{S3_BUCKET}/{OUTPUT_S3_PREFIX}"
        }
    }
)

invocation_arn = response["invocationArn"]
print(f"Job started - ARN: {invocation_arn}")

Job started - ARN: arn:aws:bedrock:us-east-1:087967639890:async-invoke/8gg8j0hpph14


## 3. Wait for Completion

In [28]:
print("Polling for job completion...")
while True:
    job = bedrock.get_async_invoke(invocationArn=invocation_arn)
    status = job["status"]
    print(f"  [{time.strftime('%H:%M:%S')}] Status: {status}")
    
    if status == "Completed":
        output_uri = job["outputDataConfig"]["s3OutputDataConfig"]["s3Uri"]
        print(f"\nJob completed! Output: {output_uri}")
        break
    elif status == "Failed":
        print(f"\nJob failed: {job.get('failureMessage', 'Unknown error')}")
        break
    
    time.sleep(15)

Polling for job completion...
  [19:03:27] Status: InProgress
  [19:03:42] Status: InProgress
  [19:03:57] Status: InProgress
  [19:04:13] Status: InProgress
  [19:04:28] Status: Completed

Job completed! Output: s3://twelvelabs-video-demo-bucket/output/marengo-multivector/8gg8j0hpph14


## 4. Download and Parse Embeddings

In [29]:
# List output files in S3
output_prefix = output_uri.replace(f"s3://{S3_BUCKET}/", "").rstrip("/")
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=output_prefix)
output_files = [obj["Key"] for obj in response.get("Contents", [])]
print(f"Output files found: {len(output_files)}")
for f in output_files:
    print(f"  - {f}")

# Download and parse output.json (Marengo outputs a single JSON file, not JSONL)
all_embeddings = []
for key in output_files:
    if key.endswith(".json") and "output" in key.split("/")[-1]:
        obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
        content = obj["Body"].read().decode("utf-8")
        parsed = json.loads(content)
        # Marengo output format: {"data": [list of embedding records]}
        if isinstance(parsed, dict) and "data" in parsed:
            all_embeddings.extend(parsed["data"])
        elif isinstance(parsed, list):
            all_embeddings.extend(parsed)

print(f"\nTotal embedding records: {len(all_embeddings)}")
if all_embeddings:
    first = all_embeddings[0]
    print(f"Record fields: {list(first.keys())}")
    print(f"Embedding dimension: {len(first['embedding'])}")

Output files found: 2
  - output/marengo-multivector/8gg8j0hpph14/manifest.json
  - output/marengo-multivector/8gg8j0hpph14/output.json

Total embedding records: 102
Record fields: ['embedding', 'embeddingOption', 'embeddingScope', 'startSec', 'endSec']
Embedding dimension: 512


## 5. Organize Embeddings by Modality and Scope

In [30]:
# Separate by modality and scope
embeddings_by_type = {
    "visual_clip": [],
    "audio_clip": [],
    "transcription_clip": [],
    "visual_asset": [],
    "audio_asset": [],
    "transcription_asset": []
}

for record in all_embeddings:
    # Marengo output: fields are directly at top level (not nested under "data")
    modality = record.get("embeddingOption", "unknown")
    scope = record.get("embeddingScope", "clip")
    key = f"{modality}_{scope}"
    
    entry = {
        "embedding": record["embedding"],
        "embeddingOption": modality,
        "embeddingScope": scope,
        "startSec": record.get("startSec", 0),
        "endSec": record.get("endSec", 0)
    }
    
    if key in embeddings_by_type:
        embeddings_by_type[key].append(entry)

# Print summary
print("Embedding Summary:")
print("=" * 50)
for key, embeds in embeddings_by_type.items():
    if embeds:
        dim = len(embeds[0]["embedding"])
        print(f"  {key}: {len(embeds)} embeddings, {dim}-dim")
    else:
        print(f"  {key}: (none)")

# Sort clip embeddings by startSec
for key in embeddings_by_type:
    if "clip" in key:
        embeddings_by_type[key].sort(key=lambda x: x["startSec"])

Embedding Summary:
  visual_clip: 33 embeddings, 512-dim
  audio_clip: 33 embeddings, 512-dim
  transcription_clip: 33 embeddings, 512-dim
  visual_asset: 1 embeddings, 512-dim
  audio_asset: 1 embeddings, 512-dim
  transcription_asset: 1 embeddings, 512-dim


In [31]:
# Save individual modality files to S3
save_map = {
    "embeddings_visual.json": embeddings_by_type["visual_clip"],
    "embeddings_audio.json": embeddings_by_type["audio_clip"],
    "embeddings_transcription.json": embeddings_by_type["transcription_clip"],
    "embeddings_asset.json": (
        embeddings_by_type["visual_asset"] + 
        embeddings_by_type["audio_asset"] + 
        embeddings_by_type["transcription_asset"]
    ),
    "embeddings_raw.json": all_embeddings
}

for filename, data in save_map.items():
    s3_key = f"{DATA_S3_PREFIX}{filename}"
    s3.put_object(
        Bucket=S3_BUCKET,
        Key=s3_key,
        Body=json.dumps(data, indent=2),
        ContentType="application/json"
    )
    print(f"💾 s3://{S3_BUCKET}/{s3_key} ({len(data)} records)")

print("\nAll embeddings saved to S3!")

💾 s3://twelvelabs-video-demo-bucket/output/marengo-multivector/embeddings/embeddings_visual.json (33 records)
💾 s3://twelvelabs-video-demo-bucket/output/marengo-multivector/embeddings/embeddings_audio.json (33 records)
💾 s3://twelvelabs-video-demo-bucket/output/marengo-multivector/embeddings/embeddings_transcription.json (33 records)
💾 s3://twelvelabs-video-demo-bucket/output/marengo-multivector/embeddings/embeddings_asset.json (3 records)
💾 s3://twelvelabs-video-demo-bucket/output/marengo-multivector/embeddings/embeddings_raw.json (102 records)

All embeddings saved to S3!


# Load and verify from S3
for modality in ["visual", "audio", "transcription"]:
    s3_key = f"{DATA_S3_PREFIX}embeddings_{modality}.json"
    obj = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    data = json.loads(obj["Body"].read().decode("utf-8"))
    
    if data:
        print(f"\n{modality.upper()} embeddings:")
        print(f"  S3 location: s3://{S3_BUCKET}/{s3_key}")
        print(f"  Segments: {len(data)}")
        print(f"  Dimension: {len(data[0]['embedding'])}")
        print(f"  Time range: {data[0]['startSec']}s - {data[-1]['endSec']}s")
        print(f"  First 5 segments:")
        for i, seg in enumerate(data[:5]):
            print(f"    [{i}] {seg['startSec']:.1f}s - {seg['endSec']:.1f}s")
    else:
        print(f"\n{modality.upper()}: No embeddings found")

# Save configuration for other notebooks
import os
config = {
    "REGION": REGION,
    "S3_BUCKET": S3_BUCKET,
    "VECTOR_BUCKET": "<YOUR-S3-VECTORS-BUCKET>",  # e.g. "my-vector-bucket"
    "MODEL_ID_SYNC": MODEL_ID_SYNC,
    "DATA_S3_PREFIX": DATA_S3_PREFIX,
    "RESULTS_S3_PREFIX": f"{OUTPUT_S3_PREFIX}results/"
}

# Save to local file
config_path = "config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"\nConfiguration saved to {config_path}")

# Also save to S3
config_key = f"{OUTPUT_S3_PREFIX}config.json"
s3.put_object(
    Bucket=S3_BUCKET,
    Key=config_key,
    Body=json.dumps(config, indent=2),
    ContentType="application/json"
)
print(f"Configuration saved to s3://{S3_BUCKET}/{config_key}")

print("\nSetup complete! Notebooks 02, 03, 04 will auto-load this configuration.")

In [ ]:
# Load and verify from S3
for modality in ["visual", "audio", "transcription"]:
    s3_key = f"{DATA_S3_PREFIX}embeddings_{modality}.json"
    obj = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    data = json.loads(obj["Body"].read().decode("utf-8"))
    
    if data:
        print(f"\n{modality.upper()} embeddings:")
        print(f"  S3 location: s3://{S3_BUCKET}/{s3_key}")
        print(f"  Segments: {len(data)}")
        print(f"  Dimension: {len(data[0]['embedding'])}")
        print(f"  Time range: {data[0]['startSec']}s - {data[-1]['endSec']}s")
        print(f"  First 5 segments:")
        for i, seg in enumerate(data[:5]):
            print(f"    [{i}] {seg['startSec']:.1f}s - {seg['endSec']:.1f}s")
    else:
        print(f"\n{modality.upper()}: No embeddings found")

# Save configuration for other notebooks
import os
config = {
    "REGION": REGION,
    "S3_BUCKET": S3_BUCKET,
    "VECTOR_BUCKET": VECTOR_BUCKET,
    "MODEL_ID_SYNC": MODEL_ID_SYNC,
    "DATA_S3_PREFIX": DATA_S3_PREFIX,
    "RESULTS_S3_PREFIX": f"{OUTPUT_S3_PREFIX}results/"
}

# Save to local file
config_path = "config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"\n💾 Configuration saved to {config_path}")

# Also save to S3
config_key = f"{OUTPUT_S3_PREFIX}config.json"
s3.put_object(
    Bucket=S3_BUCKET,
    Key=config_key,
    Body=json.dumps(config, indent=2),
    ContentType="application/json"
)
print(f"💾 Configuration saved to s3://{S3_BUCKET}/{config_key}")

print("\n✅ Setup complete! Notebooks 02, 03, 04 will auto-load this configuration.")


VISUAL embeddings:
  S3 location: s3://twelvelabs-video-demo-bucket/output/marengo-multivector/embeddings/embeddings_visual.json
  Segments: 33
  Dimension: 512
  Time range: 0.0s - 229.75s
  First 5 segments:
    [0] 0.0s - 7.5s
    [1] 7.5s - 14.2s
    [2] 14.2s - 23.8s
    [3] 23.8s - 32.8s
    [4] 32.8s - 40.0s

AUDIO embeddings:
  S3 location: s3://twelvelabs-video-demo-bucket/output/marengo-multivector/embeddings/embeddings_audio.json
  Segments: 33
  Dimension: 512
  Time range: 0.0s - 229.75s
  First 5 segments:
    [0] 0.0s - 7.5s
    [1] 7.5s - 14.2s
    [2] 14.2s - 23.8s
    [3] 23.8s - 32.8s
    [4] 32.8s - 40.0s

TRANSCRIPTION embeddings:
  S3 location: s3://twelvelabs-video-demo-bucket/output/marengo-multivector/embeddings/embeddings_transcription.json
  Segments: 33
  Dimension: 512
  Time range: 0.0s - 229.75s
  First 5 segments:
    [0] 0.0s - 7.5s
    [1] 7.5s - 14.2s
    [2] 14.2s - 23.8s
    [3] 23.8s - 32.8s
    [4] 32.8s - 40.0s

💾 Configuration saved to config.j

VECTOR_BUCKET = "<YOUR-S3-VECTORS-BUCKET>"  # Must match the value in config.json
INDEX_NAMES = ["visual", "audio", "transcription"]

s3v = boto3.client("s3vectors", region_name=REGION)

# Create indexes (skip if already exists)
for idx_name in INDEX_NAMES:
    try:
        s3v.get_index(vectorBucketName=VECTOR_BUCKET, indexName=idx_name)
        print(f"Index '{idx_name}' already exists")
    except s3v.exceptions.NotFoundException:
        s3v.create_index(
            vectorBucketName=VECTOR_BUCKET,
            indexName=idx_name,
            dataType="float32",
            dimension=512,
            distanceMetric="cosine"
        )
        print(f"Created index '{idx_name}' (512-dim, cosine)")

print("\nAll indexes ready")

In [39]:
INDEX_NAMES = ["visual", "audio", "transcription"]

s3v = boto3.client("s3vectors", region_name=REGION)

# Create indexes (skip if already exists)
for idx_name in INDEX_NAMES:
    try:
        s3v.get_index(vectorBucketName=VECTOR_BUCKET, indexName=idx_name)
        print(f"Index '{idx_name}' already exists")
    except s3v.exceptions.NotFoundException:
        s3v.create_index(
            vectorBucketName=VECTOR_BUCKET,
            indexName=idx_name,
            dataType="float32",
            dimension=512,
            distanceMetric="cosine"
        )
        print(f"Created index '{idx_name}' (512-dim, cosine)")

print("\nAll indexes ready")

Index 'visual' already exists
Index 'audio' already exists
Index 'transcription' already exists

All indexes ready


In [40]:
# Store embeddings in S3 Vectors (batch of up to 50 at a time)
BATCH_SIZE = 50

for modality in INDEX_NAMES:
    # Load from S3
    s3_key = f"{DATA_S3_PREFIX}embeddings_{modality}.json"
    obj = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    data = json.loads(obj["Body"].read().decode("utf-8"))
    
    total = 0
    for i in range(0, len(data), BATCH_SIZE):
        batch = data[i:i+BATCH_SIZE]
        vectors = []
        for seg in batch:
            vectors.append({
                "key": f"{modality}_{seg['startSec']:.1f}_{seg['endSec']:.1f}",
                "data": {"float32": [float(x) for x in seg["embedding"]]},
                "metadata": {
                    "startSec": seg["startSec"],
                    "endSec": seg["endSec"],
                    "modality": modality
                }
            })
        s3v.put_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=modality,
            vectors=vectors
        )
        total += len(vectors)
    
    print(f"Stored {total} vectors in '{modality}' index")

print("\nAll embeddings stored in S3 Vectors!")

Stored 33 vectors in 'visual' index
Stored 33 vectors in 'audio' index
Stored 33 vectors in 'transcription' index

All embeddings stored in S3 Vectors!
